In [ ]:
import sys, os
# Adjust depth based on notebook location relative to project root
_src_path = os.path.normpath(os.path.join(os.getcwd(), '..', '..', 'src'))
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)
from project_config import load_config

config    = load_config()
base_dir  = config['base_dir']
well_list = config['well_list']
well_meta = config.get('well_metadata', {})

print(f'Active experiment: {config.get("experiment_name", config.get("experiment_key", "?"))}')
print(f'Base dir: {base_dir}')

In [ ]:
# Cell 1: imports & config (adds PCA + 3D)
import sys, os, re, io, gc, time
import numpy as np
import pandas as pd
from itertools import combinations
from datetime import datetime

from scipy import sparse
from scipy.stats import spearmanr, pearsonr
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans as KMeans
from sklearn.metrics import silhouette_score

from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.shapes import MSO_AUTO_SHAPE_TYPE
from pptx.dml.color import RGBColor

import anndata as ad

# optional (for LOWESS curve)
try:
    from statsmodels.nonparametric.smoothers_lowess import lowess
    HAS_LOWESS = True
except Exception:
    HAS_LOWESS = False

print("OK: imports loaded")

# === Input / Output ===
H5AD_PATH = r"D:\Sam\4i_Liposarcoma_Data_YR2025\LPS_246_Abema_July_2025\standard_adata_sub.h5ad"
assert os.path.exists(H5AD_PATH), f"Not found: {H5AD_PATH}"

OUTPUT_PPTX = os.path.join(
    os.path.dirname(H5AD_PATH),
    os.path.splitext(os.path.basename(H5AD_PATH))[0] + "_FULL_pairs.pptx"
)

# === Treatment grouping ===
OBS_SPLIT_KEY = "treatment"   # or "sample_ID"

# === Feature aliases (compare one-by-one)
FEATURE_ALIASES = [
    "pRb","Rb","CDK2","CDK4","p53","p21",
    "cycD1","Mdm2","cycB1","Ki67","p16","cycA2",
    "pS6","S6","Cdt1","E2F1"
]
USE_FEATURE_ALIASES_ONLY = True

# === Exclude any features globally (regex)
EXCLUDE_FEATURE_PATTERNS = [r"_DNA_nuc_mean$"]  # drop *_DNA_nuc_mean

# === Rendering / behavior knobs ===
FIG_DPI        = 110
SAVE_EVERY     = 20
RANDOM_STATE   = 42
MAX_TREATMENT_FACETS = 8
KMEANS_K_RANGE = range(2, 7)   # for "all cells" clustering on each pair

# === Chunking
START_IDX = 0
END_IDX   = 500

PALETTE = ['#7E57C2', '#FFCA28', '#26A69A', '#EF5350', '#42A5F5', '#8D6E63', '#AB47BC', '#FF7043']
AXIS_PT, LEGEND_PT = 9, 11


OK: imports loaded


In [18]:
# Cell 2: helpers (colorbar/legend positions, captions, compare builders, KMeans+3D)

from pptx.enum.text import PP_ALIGN

def stamp(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

def clean_dense_array(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    X[~np.isfinite(X)] = np.nan
    med = np.nanmedian(X, axis=0); med[~np.isfinite(med)] = 0.0
    r, c = np.where(~np.isfinite(X))
    if r.size: X[r, c] = med[c]
    hi = np.nanquantile(X, 0.999, axis=0)
    return np.minimum(X, hi)

def benjamini_hochberg(pvals: np.ndarray) -> np.ndarray:
    p = np.asarray(pvals, dtype=float); n = p.size
    order = np.argsort(p)
    rank = np.empty_like(order); rank[order] = np.arange(1, n+1)
    q = p * n / rank
    q_sorted = np.minimum.accumulate(q[order][::-1])[::-1]
    out = np.empty_like(q_sorted); out[order] = q_sorted
    return np.clip(out, 0, 1)

def set_title(slide, main, sub):
    tf = slide.shapes.title.text_frame
    tf.clear()
    p0 = tf.paragraphs[0]; p0.text = main
    for r in p0.runs: r.font.size = Pt(24); r.font.bold = True
    p1 = tf.add_paragraph(); p1.text = sub
    for r in p1.runs: r.font.size = Pt(14)

def hex_to_rgb(s):
    s = s.lstrip('#'); return tuple(int(s[i:i+2], 16) for i in (0,2,4))

def map_aliases_to_vars(feature_names, aliases):
    """
    Map each alias (e.g., 'pRb') to a var name in feature_names.
    Preference: contains alias, and among matches prefer:
      'nuc_mean' > 'total_nuc_protein' > 'nuc_cyto_ratio' > shortest name.
    """
    fn = pd.Series(feature_names, dtype=str)
    lower_fn = fn.str.lower()

    mapping = {}
    for a in aliases:
        a_low = a.lower()

        # 1) regex: whole-token-ish match (^|_)alias(_|$)
        pat = re.compile(rf'(^|_){re.escape(a_low)}(_|$)', re.IGNORECASE)
        mask = lower_fn.str.contains(pat, na=False)
        hits = fn[mask]

        # 2) fallback: simple substring (case-insensitive)
        if hits.empty:
            mask2 = lower_fn.str.contains(re.escape(a_low), regex=True, na=False)
            hits = fn[mask2]

        if not hits.empty:
            pref = ['nuc_mean', 'total_nuc_protein', 'nuc_cyto_ratio']
            def score(name: str):
                n = name.lower()
                for i, key in enumerate(pref):
                    if key in n:
                        return (i, len(n))
                return (len(pref), len(n))
            chosen = sorted(hits, key=score)[0]
            mapping[a] = str(chosen)
        else:
            mapping[a] = None  # no match in the var names

    return mapping


def add_table_from_df(slide, df, left, top, width, height, col_fmt=None):
    rows, cols = df.shape[0]+1, df.shape[1]
    shape = slide.shapes.add_table(rows, cols, left, top, width, height)
    table = shape.table
    for j, col in enumerate(df.columns):
        cell = table.cell(0, j)
        cell.text = str(col)
        p = cell.text_frame.paragraphs[0]
        p.runs[0].font.bold = True
        p.alignment = PP_ALIGN.CENTER
    for i in range(df.shape[0]):
        for j in range(cols):
            val = df.iat[i, j]
            cell_text = col_fmt[j](val) if (col_fmt and j in col_fmt) else ("" if pd.isna(val) else str(val))
            cell = table.cell(i+1, j)
            cell.text = cell_text
            cell.text_frame.paragraphs[0].alignment = PP_ALIGN.CENTER
    return table

def fig_to_buf(fig):
    b = io.BytesIO()
    fig.savefig(b, format="png", bbox_inches="tight", dpi=FIG_DPI)
    plt.close(fig); b.seek(0)
    return b

def hexbin_panel(ax, x, y, title, xlabel, ylabel):
    hb = ax.hexbin(x, y, gridsize=45, mincnt=1, cmap="viridis")
    ax.set_title(title, fontsize=AXIS_PT+1)
    ax.set_xlabel(xlabel, fontsize=AXIS_PT)
    ax.set_ylabel(ylabel, fontsize=AXIS_PT)
    ax.tick_params(labelsize=AXIS_PT)
    if HAS_LOWESS:
        try:
            curve = lowess(y, x, frac=0.2, return_sorted=True)
            ax.plot(curve[:,0], curve[:,1], linewidth=1.2)
        except Exception:
            pass
    return hb

def caption(fig, text):
    fig.text(0.01, 0.01, text, fontsize=8, ha="left", va="bottom")

# ---- comparison figure (side-by-side) with *bottom* shared colorbar + caption
def make_pairwise_compare_buffer(fa, fb, left_treat, right_treat, X, feature_names, group_idx):
    ia = int(np.where(feature_names == fa)[0][0]); ib = int(np.where(feature_names == fb)[0][0])
    fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.8), dpi=FIG_DPI)
    hbs = []
    for ax, g in zip(axes, [left_treat, right_treat]):
        idxs = group_idx[g]
        xa, xb = X[idxs, ia], X[idxs, ib]
        hbs.append(hexbin_panel(ax, xa, xb, title=g, xlabel=fa, ylabel=fb))
    # put colorbar UNDER the panels so it never overlaps
    cax = fig.add_axes([0.13, 0.08, 0.74, 0.03])  # [left, bottom, width, height]
    cb = fig.colorbar(hbs[0], cax=cax, orientation='horizontal')
    cb.set_label("Cell count per hexagon", fontsize=AXIS_PT)
    fig.suptitle(f"{fa} vs {fb} — {left_treat} vs {right_treat}", fontsize=AXIS_PT+3)
    caption(fig, "Plot type: hexbin density scatter. Colors = # of cells per hexagon. Line = LOWESS trend.")
    fig.tight_layout(rect=[0, 0.12, 1, 0.92])  # leave room for bottom colorbar and title
    return fig_to_buf(fig)

# ---- All-cells: KMeans (2D) + 3D, on ONE figure; legends moved outside
def choose_best_kmeans_labels(X_pair, k_range=range(2,7), random_state=42):
    from sklearn.cluster import MiniBatchKMeans as KMeans
    from sklearn.metrics import silhouette_score
    best_k, best_sil, best_lab = None, -np.inf, None
    for k in k_range:
        km = KMeans(n_clusters=k, n_init=5, random_state=random_state, batch_size=2048)
        lab = km.fit_predict(X_pair)
        if np.unique(lab).size < 2: 
            continue
        sil = silhouette_score(X_pair, lab)
        if sil > best_sil:
            best_k, best_sil, best_lab = k, sil, lab
    if best_lab is None:
        km = KMeans(n_clusters=2, n_init=5, random_state=random_state, batch_size=2048)
        best_lab = km.fit_predict(X_pair); best_k, best_sil = 2, float('nan')
    return best_k, best_sil, best_lab

def make_kmeans3d_combo_buffer(fa, fb, X, feature_names, split_vals, pc1_all):
    ia = int(np.where(feature_names == fa)[0][0]); ib = int(np.where(feature_names == fb)[0][0])
    XY = np.column_stack([X[:, ia], X[:, ib]])
    k, sil, labels = choose_best_kmeans_labels(XY, k_range=KMEANS_K_RANGE, random_state=RANDOM_STATE)

    fig = plt.figure(figsize=(9.2, 3.8), dpi=FIG_DPI)
    # left: KMeans 2D
    ax1 = fig.add_subplot(1,2,1)
    uniq = np.sort(np.unique(labels))
    for i, c in enumerate(uniq):
        m = (labels == c)
        ax1.scatter(XY[m,0], XY[m,1], s=6, color=PALETTE[i % len(PALETTE)], edgecolors='none', alpha=0.9, label=f"C{c} (n={m.sum():,})")
    ax1.set_xlabel(fa, fontsize=AXIS_PT); ax1.set_ylabel(fb, fontsize=AXIS_PT)
    ax1.set_title(f"KMeans (best K={k}, silhouette={sil:.3f})", fontsize=AXIS_PT+2)
    # legend OUTSIDE panel
    ax1.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8, frameon=False)

    # right: 3D with treatment legend OUTSIDE
    ax2 = fig.add_subplot(1,2,2, projection='3d')
    treats = pd.Categorical(split_vals).categories
    for i, g in enumerate(treats):
        m = (split_vals == g)
        ax2.scatter(X[m, ia], X[m, ib], pc1_all[m], s=4, alpha=0.8, label=f"{g} (n={m.sum():,})")
    ax2.set_xlabel(fa); ax2.set_ylabel(fb); ax2.set_zlabel("PC1 (all features)")
    ax2.set_title("3D view (colored by treatment)")
    # move legend BELOW plot area, centered
    fig.legend(loc='lower center', bbox_to_anchor=(0.5, 0.02), ncol=min(len(treats), 4), fontsize=8, frameon=False)

    caption(fig, "Left: 2D scatter colored by KMeans clusters (all treatments pooled). Right: 3D scatter; Z = PC1 across all selected features.")
    fig.tight_layout(rect=[0.03, 0.08, 0.97, 0.92])
    return fig_to_buf(fig)


In [19]:
# Cell 3: load AnnData, select features, split groups, compute PC1 for 3D

stamp(f"Loading AnnData: {H5AD_PATH}")
adata = ad.read_h5ad(H5AD_PATH)
X = adata.X.toarray() if sparse.issparse(adata.X) else np.asarray(adata.X, dtype=float)
X = clean_dense_array(X)
all_var = np.array(adata.var_names, dtype=str)

# Exclude patterns first
if EXCLUDE_FEATURE_PATTERNS:
    pat = re.compile("|".join(EXCLUDE_FEATURE_PATTERNS))
    keep_mask = np.array([not bool(pat.search(v)) for v in all_var])
    removed = all_var[~keep_mask].tolist()
    if removed:
        stamp(f"Excluding {len(removed)} features: {removed[:10]}{' ...' if len(removed)>10 else ''}")
    all_var = all_var[keep_mask]; X = X[:, keep_mask]

# Map aliases -> chosen var names
alias_map = map_aliases_to_vars(all_var, FEATURE_ALIASES)
print("Alias → var mapping:")
for a, v in alias_map.items():
    print(f"  {a:>5}  →  {v}")

if USE_FEATURE_ALIASES_ONLY:
    chosen = [alias_map[a] for a in FEATURE_ALIASES if alias_map[a] is not None]
    missing = [a for a in FEATURE_ALIASES if alias_map[a] is None]
    if missing:
        print("WARNING: no column for:", missing)
    feature_names = np.array(chosen, dtype=str)
    idx = [np.where(all_var == n)[0][0] for n in feature_names]
    X = X[:, idx]
else:
    feature_names = all_var.copy()

print(f"Using {len(feature_names)} features:", list(feature_names))

# Groups
if OBS_SPLIT_KEY in adata.obs.columns:
    split_vals = adata.obs[OBS_SPLIT_KEY].astype(str).values
else:
    split_vals = np.array(["all"] * adata.n_obs)
cats = pd.Categorical(split_vals).categories
group_idx = {g: np.flatnonzero(split_vals == g) for g in cats}
if not group_idx:
    group_idx = {"all": np.arange(adata.n_obs)}
stamp("Groups: " + ", ".join(f"{g}={len(idx)}" for g, idx in group_idx.items()))

# PC1 across all selected features (for 3D z-axis)
X_mean = np.nanmean(X, axis=0); X_std = np.nanstd(X, axis=0); X_std[X_std == 0] = 1.0
Z_all = (X - X_mean) / X_std
pc1_all = PCA(n_components=1, random_state=RANDOM_STATE).fit_transform(np.nan_to_num(Z_all)).ravel()


[10:45:17] Loading AnnData: D:\Sam\4i_Liposarcoma_Data_YR2025\LPS_246_Abema_July_2025\standard_adata_sub_sub.h5ad
[10:45:17] Excluding 6 features: ['R0_DNA_nuc_mean', 'R1_DNA_nuc_mean', 'R2_DNA_nuc_mean', 'R3_DNA_nuc_mean', 'R4_DNA_nuc_mean', 'R5_DNA_nuc_mean']
Alias → var mapping:
    pRb  →  R0_pRb_nuc_mean
     Rb  →  R0_Rb_nuc_mean
   CDK2  →  R0_CDK2_nuc_mean
   CDK4  →  R1_CDK4_nuc_mean
    p53  →  R1_p53_nuc_mean
    p21  →  R1_p21_nuc_mean
  cycD1  →  R2_cycD1_nuc_mean
   Mdm2  →  R2_Mdm2_nuc_mean
  cycB1  →  R2_cycB1_nuc_mean
   Ki67  →  R3_Ki67_nuc_mean
    p16  →  R3_p16_nuc_mean
  cycA2  →  R3_cycA2_nuc_mean
    pS6  →  R4_pS6_nuc_mean
     S6  →  R4_S6_nuc_mean
   Cdt1  →  R5_Cdt1_nuc_mean
   E2F1  →  R5_E2F1_nuc_mean
Using 16 features: ['R0_pRb_nuc_mean', 'R0_Rb_nuc_mean', 'R0_CDK2_nuc_mean', 'R1_CDK4_nuc_mean', 'R1_p53_nuc_mean', 'R1_p21_nuc_mean', 'R2_cycD1_nuc_mean', 'R2_Mdm2_nuc_mean', 'R2_cycB1_nuc_mean', 'R3_Ki67_nuc_mean', 'R3_p16_nuc_mean', 'R3_cycA2_nuc_mean', 'R

c:\Users\Purvis_Jane\.conda\envs\GPU_Segment_4i\lib\site-packages\ipykernel_launcher.py:52: UserWarning: This pattern has match groups. To actually get the groups, use str.extract.


In [20]:
# Cell 4: compute per-treatment correlations (Spearman, Pearson) + BH-FDR

stamp("Computing per-treatment correlations…")
pairs = [(feature_names[i], feature_names[j]) for i, j in combinations(range(len(feature_names)), 2)]

rows = []
for g, idxs in group_idx.items():
    if idxs.size < 3:
        continue
    pvals = []; tmp = []
    for (fa, fb) in pairs:
        ia = int(np.where(feature_names == fa)[0][0])
        ib = int(np.where(feature_names == fb)[0][0])
        xa, xb = X[idxs, ia], X[idxs, ib]
        n = len(xa)
        if np.nanstd(xa) == 0 or np.nanstd(xb) == 0 or n < 3:
            rho, p, r_lin = np.nan, 1.0, np.nan
        else:
            rho, p = spearmanr(xa, xb)
            try:
                r_lin, _ = pearsonr(xa, xb)
            except Exception:
                r_lin = np.nan
        pvals.append(p)
        tmp.append((g, fa, fb, rho, r_lin, p, n))
    q = benjamini_hochberg(np.array(pvals, dtype=float))
    for (g, fa, fb, rho, r_lin, p, n), qq in zip(tmp, q):
        rows.append((g, fa, fb, rho, r_lin, p, qq, n))

by_treat_df = pd.DataFrame(rows, columns=["treatment","feat_a","feat_b","rho_spearman","r_pearson","pval","fdr","n"])

# Rank pairs by min FDR across treatments; tie-breaker: max |rho| across treatments
rank_df = (by_treat_df
           .groupby(["feat_a","feat_b"], as_index=False)
           .agg(min_fdr=("fdr","min"),
                max_abs_rho=("rho_spearman", lambda x: np.nanmax(np.abs(x)))))
rank_df = rank_df.sort_values(["min_fdr","max_abs_rho"], ascending=[True, False]).reset_index(drop=True)

print("Top 10 pairs by best FDR:")
display(rank_df.head(10))

# For quick lookup of per-pair stats
pair_stats = { (fa,fb): sub.sort_values("treatment").reset_index(drop=True)
               for (fa,fb), sub in by_treat_df.groupby(["feat_a","feat_b"]) }
pair_stats.update({ (fb,fa): sub for (fa,fb), sub in list(pair_stats.items())})  # symmetry

# Slice the chunk to render
full_count = len(rank_df)
sel_df = rank_df.iloc[START_IDX:END_IDX].reset_index(drop=True)
stamp(f"Rendering chunk {START_IDX}:{min(END_IDX, full_count)} (n={len(sel_df)})")


[10:45:20] Computing per-treatment correlations…
Top 10 pairs by best FDR:


,feat_a,feat_b,min_fdr,max_abs_rho
0,R3_p16_nuc_mean,R3_cycA2_nuc_mean,0.0,0.891283
1,R0_pRb_nuc_mean,R0_Rb_nuc_mean,0.0,0.876088
2,R0_Rb_nuc_mean,R0_CDK2_nuc_mean,0.0,0.875880
3,R1_p21_nuc_mean,R2_cycD1_nuc_mean,0.0,0.862127
4,R4_pS6_nuc_mean,R4_S6_nuc_mean,0.0,0.777582
5,R2_Mdm2_nuc_mean,R2_cycB1_nuc_mean,0.0,0.772948
6,R1_p53_nuc_mean,R1_p21_nuc_mean,0.0,0.766933
7,R0_CDK2_nuc_mean,R2_cycB1_nuc_mean,0.0,0.735276
8,R0_Rb_nuc_mean,R2_cycB1_nuc_mean,0.0,0.724429
9,R3_p16_nuc_mean,R4_S6_nuc_mean,0.0,0.691681


[10:45:21] Rendering chunk 0:120 (n=120)


In [21]:
# Cell 5: init PPT, title, TOC, and global heatmaps by treatment (+ delta vs control if present)

prs = Presentation()
prs.core_properties.title = "Feature Correlations by Treatment"

# Title slide
title = prs.slides.add_slide(prs.slide_layouts[0])
set_title(
    title,
    f"{os.path.basename(H5AD_PATH)} — Feature Correlations",
    f"Per-treatment Spearman (BH-FDR) + Pearson | chunk {START_IDX}:{END_IDX} of {full_count}"
)

# TOC slide
toc = prs.slides.add_slide(prs.slide_layouts[1])
toc.shapes.title.text = "Top Pairs (this chunk)"
tf = toc.shapes.placeholders[1].text_frame; tf.clear()
for _, r in sel_df.iterrows():
    p = tf.add_paragraph()
    p.text = f"{r['feat_a']} vs {r['feat_b']} | min FDR={r['min_fdr']:.2e} | max |ρ|={r['max_abs_rho']:.3f}"
    p.level = 1

# === Global correlation heatmaps per treatment ===
stamp("Rendering global correlation heatmaps per treatment…")
feat_list = list(feature_names)
feat_index = {f:i for i,f in enumerate(feature_names)}

# Function to compute Spearman matrix given indices
def spearman_mat(subX):
    m = subX.shape[1]
    R = np.full((m,m), np.nan, dtype=float)
    for i in range(m):
        R[i,i] = 1.0
        for j in range(i+1, m):
            xi, xj = subX[:,i], subX[:,j]
            if np.nanstd(xi)==0 or np.nanstd(xj)==0 or len(xi)<3:
                r = np.nan
            else:
                r, _ = spearmanr(xi, xj)
            R[i,j] = R[j,i] = r
    return R

treat_to_R = {}
for g, idxs in group_idx.items():
    if idxs.size < 3: 
        continue
    R = spearman_mat(X[idxs,:])
    treat_to_R[g] = R

# Heatmap slide(s)
def heatmap_buf(R, title):
    fig = plt.figure(figsize=(6.5, 5.6), dpi=FIG_DPI)
    ax = plt.gca()
    vlim = np.nanmax(np.abs(R[np.isfinite(R)])) if np.isfinite(R).any() else 1.0
    im = ax.imshow(R, vmin=-vlim, vmax=vlim, cmap="coolwarm", interpolation="nearest")
    ax.set_xticks(range(len(feat_list))); ax.set_xticklabels(feat_list, rotation=90, fontsize=8)
    ax.set_yticks(range(len(feat_list))); ax.set_yticklabels(feat_list, fontsize=8)
    ax.set_title(title, fontsize=AXIS_PT+3)
    cb = plt.colorbar(im, fraction=0.046, pad=0.04)
    cb.set_label("Spearman ρ", fontsize=AXIS_PT)
    return fig_to_buf(fig)

# One slide: multiple treatments (up to 4 per slide)
treatments = list(treat_to_R.keys())
for i in range(0, len(treatments), 4):
    slide = prs.slides.add_slide(prs.slide_layouts[5])  # Title Only
    seg = treatments[i:i+4]
    set_title(slide, "Global correlation heatmaps", f"Treatments {i+1}-{i+len(seg)} of {len(treatments)}")
    lefts = [Inches(0.3), Inches(5.1)]
    tops  = [Inches(1.3), Inches(4.6)]
    idxpos = [(0,0),(1,0),(0,1),(1,1)]
    for k, g in enumerate(seg):
        r, c = idxpos[k]
        buf = heatmap_buf(treat_to_R[g], f"{g} (Spearman)")
        slide.shapes.add_picture(buf, lefts[r], tops[c], width=Inches(4.5))

# Optional Δ vs DMSO if present
if "DMSO" in treat_to_R:
    base = treat_to_R["DMSO"]
    for g, R in treat_to_R.items():
        if g == "DMSO": 
            continue
        Delta = R - base
        fig = plt.figure(figsize=(6.5, 5.6), dpi=FIG_DPI)
        ax = plt.gca()
        vmax = np.nanpercentile(np.abs(Delta[np.isfinite(Delta)]), 99) if np.isfinite(Delta).any() else 0.5
        im = ax.imshow(Delta, cmap="bwr", norm=TwoSlopeNorm(vmin=-vmax, vcenter=0., vmax=vmax))
        ax.set_xticks(range(len(feat_list))); ax.set_xticklabels(feat_list, rotation=90, fontsize=8)
        ax.set_yticks(range(len(feat_list))); ax.set_yticklabels(feat_list, fontsize=8)
        ax.set_title(f"Δ Spearman vs DMSO — {g}", fontsize=AXIS_PT+3)
        cb = plt.colorbar(im, fraction=0.046, pad=0.04); cb.set_label("Δρ", fontsize=AXIS_PT)
        buf = fig_to_buf(fig)
        slide = prs.slides.add_slide(prs.slide_layouts[5])
        set_title(slide, f"Δ correlation heatmap: {g}", "Δ = ρ(g) − ρ(DMSO)")
        slide.shapes.add_picture(buf, Inches(0.6), Inches(1.6), width=Inches(9.0))


[10:45:36] Rendering global correlation heatmaps per treatment…


In [22]:
# Cell 6: per-pair slides — pairwise comparisons only + one KMeans+3D slide per pair

def two_treat_stats_table(stats_sub, t1, t2):
    return (stats_sub
            .loc[stats_sub['treatment'].isin([t1, t2]),
                 ["treatment","rho_spearman","r_pearson","pval","fdr","n"]]
            .sort_values("treatment")
            .reset_index(drop=True))

col_fmt = {
    1: lambda v: "" if pd.isna(v) else f"{v:.3f}",
    2: lambda v: "" if pd.isna(v) else f"{v:.3f}",
    3: lambda v: "" if pd.isna(v) else f"{v:.2e}",
    4: lambda v: "" if pd.isna(v) else f"{v:.2e}",
    5: lambda v: "" if pd.isna(v) else f"{int(v):,}"
}

from time import perf_counter
t0 = perf_counter()
n = len(sel_df)

present = set(group_idx.keys())
want_pairs = [("Abema","Control"), ("Nutlin","Control"), ("Abema","Nutlin")]
want_pairs = [p for p in want_pairs if set(p).issubset(present)]

print("Will render", n, "feature pairs; comparisons per pair:", want_pairs)
print("Saving to:", OUTPUT_PPTX)

for idx, rr in sel_df.iterrows():
    fa, fb = rr["feat_a"], rr["feat_b"]
    stats_sub = pair_stats.get((fa, fb), None)
    if stats_sub is None or stats_sub.empty:
        continue

    # === Slide 1: All-cells KMeans + 3D (single slide, legends/cbar placed cleanly)
    combo = make_kmeans3d_combo_buffer(fa, fb, X, feature_names, split_vals, pc1_all)
    slide_combo = prs.slides.add_slide(prs.slide_layouts[5])
    set_title(slide_combo, f"{fa} vs {fb} — All cells", "Left: KMeans clusters • Right: 3D view colored by treatment")
    slide_combo.shapes.add_picture(combo, Inches(0.6), Inches(1.4), width=Inches(9.0))

    # === Slides 2..: requested pairwise comparisons, each with bottom colorbar + caption and a tidy 2-row stats table
    for (left_t, right_t) in want_pairs:
        buf_cmp = make_pairwise_compare_buffer(fa, fb, left_t, right_t, X, feature_names, group_idx)
        slide = prs.slides.add_slide(prs.slide_layouts[5])
        set_title(slide, f"{fa} vs {fb} — {left_t} vs {right_t}",
                  "Hexbin density with shared bottom colorbar; LOWESS trend per panel")
        # place the comparison figure left; put table to the right with comfortable margins
        slide.shapes.add_picture(buf_cmp, Inches(0.6), Inches(1.4), width=Inches(7.0))
        tbl_df = two_treat_stats_table(stats_sub, left_t, right_t)
        add_table_from_df(
            slide, tbl_df,
            left=Inches(7.8), top=Inches(1.4),
            width=Inches(2.0), height=Inches(2.8),
            col_fmt=col_fmt
        )

    # ---- checkpoint & progress
    if (idx + 1) % SAVE_EVERY == 0:
        prs.save(OUTPUT_PPTX)
        stamp(f"Checkpoint saved at {idx+1}/{n} → {OUTPUT_PPTX}")

    avg = (perf_counter() - t0) / (idx + 1)
    print(f"pair {idx+1}/{n} | avg {avg:.2f}s/pair")


Will render 120 feature pairs; comparisons per pair: [('Abema', 'Control'), ('Nutlin', 'Control'), ('Abema', 'Nutlin')]
Saving to: D:\Sam\4i_Liposarcoma_Data_YR2025\LPS_246_Abema_July_2025\standard_adata_sub_sub_FULL_pairs.pptx


c:\Users\Purvis_Jane\.conda\envs\GPU_Segment_4i\lib\site-packages\ipykernel_launcher.py:133: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.


pair 1/120 | avg 8.52s/pair


c:\Users\Purvis_Jane\.conda\envs\GPU_Segment_4i\lib\site-packages\ipykernel_launcher.py:133: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.


pair 2/120 | avg 8.27s/pair


...

pair 118/120 | avg 8.15s/pair


c:\Users\Purvis_Jane\.conda\envs\GPU_Segment_4i\lib\site-packages\ipykernel_launcher.py:133: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.


pair 119/120 | avg 8.15s/pair


c:\Users\Purvis_Jane\.conda\envs\GPU_Segment_4i\lib\site-packages\ipykernel_launcher.py:133: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.


[11:02:09] Checkpoint saved at 120/120 → D:\Sam\4i_Liposarcoma_Data_YR2025\LPS_246_Abema_July_2025\standard_adata_sub_sub_FULL_pairs.pptx
pair 120/120 | avg 8.16s/pair


In [23]:
# Cell 7: save & quick checks

prs.save(OUTPUT_PPTX)
stamp(f"Saved deck to: {OUTPUT_PPTX}")

print("OUTPUT_PPTX =", OUTPUT_PPTX)
print("Absolute path:", os.path.abspath(OUTPUT_PPTX))
print("Exists?", os.path.exists(OUTPUT_PPTX))
if os.path.exists(OUTPUT_PPTX):
    print("Size (bytes):", os.path.getsize(OUTPUT_PPTX))
    try:
        os.startfile(os.path.dirname(OUTPUT_PPTX))
    except Exception as e:
        print("Could not open File Explorer:", e)

# Optional: one-slide smoke test for the very first pair in sel_df (adds an extra slide)
if len(sel_df) > 0:
    rr = sel_df.iloc[0]
    fa, fb = rr["feat_a"], rr["feat_b"]
    stats_sub = pair_stats[(fa, fb)]
    facet_treats = list(stats_sub.sort_values("n", ascending=False)["treatment"].head(min(6, len(stats_sub))))
    buf_multi = per_pair_small_multiples(fa, fb, facet_treats)

    slide = prs.slides.add_slide(prs.slide_layouts[5])
    set_title(slide, f"{fa} vs {fb} — smoke test", f"Top {len(facet_treats)} treatments by n")
    slide.shapes.add_picture(buf_multi, Inches(0.6), Inches(1.6), width=Inches(9.0))
    prs.save(OUTPUT_PPTX)
    print("Smoke test slide appended.")


[11:02:42] Saved deck to: D:\Sam\4i_Liposarcoma_Data_YR2025\LPS_246_Abema_July_2025\standard_adata_sub_sub_FULL_pairs.pptx
OUTPUT_PPTX = D:\Sam\4i_Liposarcoma_Data_YR2025\LPS_246_Abema_July_2025\standard_adata_sub_sub_FULL_pairs.pptx
Absolute path: D:\Sam\4i_Liposarcoma_Data_YR2025\LPS_246_Abema_July_2025\standard_adata_sub_sub_FULL_pairs.pptx
Exists? True
Size (bytes): 52496640
Smoke test slide appended.


In [34]:
import os

print("OUTPUT_PPTX =", OUTPUT_PPTX)
print("Absolute path:", os.path.abspath(OUTPUT_PPTX))
print("Exists?", os.path.exists(OUTPUT_PPTX))

if os.path.exists(OUTPUT_PPTX):
    print("Size (bytes):", os.path.getsize(OUTPUT_PPTX))
    # Open the folder in File Explorer
    try:
        os.startfile(os.path.dirname(OUTPUT_PPTX))
    except Exception as e:
        print("Could not open File Explorer:", e)


OUTPUT_PPTX = D:\Sam\4i_Liposarcoma_Data_YR2025\LPS_246_Abema_July_2025\standard_adata_sub_sub_pairs_by_treatment.pptx
Absolute path: D:\Sam\4i_Liposarcoma_Data_YR2025\LPS_246_Abema_July_2025\standard_adata_sub_sub_pairs_by_treatment.pptx
Exists? False


In [35]:
from datetime import datetime
import os

try:
    out2 = os.path.join(
        os.path.dirname(H5AD_PATH),
        f"pairs_by_treatment_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pptx"
    )
    prs.save(out2)
    print("Resaved to:", out2)
    try:
        os.startfile(out2)  # open the file directly
    except Exception as e:
        print("Open failed:", e)
except NameError:
    print("It looks like the 'prs' object is not in memory. Re-run Cells 6–7 to rebuild, then run this cell again.")


Resaved to: D:\Sam\4i_Liposarcoma_Data_YR2025\LPS_246_Abema_July_2025\pairs_by_treatment_20250911_165549.pptx


In [36]:
print("len(rank_df) total =", len(rank_df))
print("Chunk indices:", START_IDX, END_IDX)
print("len(sel_df) chunk =", len(sel_df))
print(sel_df.head(3))


len(rank_df) total = 120
Chunk indices: 0 120
len(sel_df) chunk = 120
            feat_a             feat_b  min_fdr  max_abs_rho
0  R3_p16_nuc_mean  R3_cycA2_nuc_mean      0.0     0.891283
1  R0_pRb_nuc_mean     R0_Rb_nuc_mean      0.0     0.876088
2   R0_Rb_nuc_mean   R0_CDK2_nuc_mean      0.0     0.875880


In [37]:
import os, time
print("OUTPUT_PPTX =", OUTPUT_PPTX)
print("Exists?", os.path.exists(OUTPUT_PPTX))
if os.path.exists(OUTPUT_PPTX):
    print("Size:", os.path.getsize(OUTPUT_PPTX), "bytes")
    print("Modified:", time.ctime(os.path.getmtime(OUTPUT_PPTX)))
    os.startfile(OUTPUT_PPTX)  # opens the exact file


OUTPUT_PPTX = D:\Sam\4i_Liposarcoma_Data_YR2025\LPS_246_Abema_July_2025\standard_adata_sub_sub_pairs_by_treatment.pptx
Exists? False


In [38]:
# Smoke test: build exactly one slide from the first row of sel_df
assert len(sel_df) > 0, "No pairs in sel_df — adjust START_IDX/END_IDX."
rr = sel_df.iloc[0]
fa, fb = rr["feat_a"], rr["feat_b"]
ia = int(np.where(feature_names == fa)[0][0]); ib = int(np.where(feature_names == fb)[0][0])
xa, xb = X[:, ia], X[:, ib]
XYz = Z[:, [ia, ib]]

from sklearn.cluster import MiniBatchKMeans as KMeans
from sklearn.metrics import silhouette_score
PALETTE = ['#7E57C2', '#FFCA28', '#26A69A', '#EF5350', '#42A5F5', '#8D6E63', '#AB47BC', '#FF7043']
RANDOM_STATE = 42; FIG_DPI = 110; AXIS_PT = 9; LEGEND_PT = 11
K_RANGE = range(2, 4)

# pick best K
best_k, best_sil, labels = None, -np.inf, None
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=5, random_state=RANDOM_STATE, batch_size=2048)
    lab = km.fit_predict(XYz)
    if np.unique(lab).size < 2: continue
    sil = silhouette_score(XYz, lab)
    if sil > best_sil: best_k, best_sil, labels = k, sil, lab
if labels is None:
    km = KMeans(n_clusters=2, n_init=5, random_state=RANDOM_STATE, batch_size=2048)
    labels = km.fit_predict(XYz); best_k, best_sil = 2, float('nan')

# plots in-memory
import io, matplotlib.pyplot as plt
buf_sc = io.BytesIO()
fig = plt.figure(figsize=(3.0, 2.8), dpi=FIG_DPI); ax = plt.gca()
for i,c in enumerate(sorted(np.unique(labels))):
    m = (labels==c)
    ax.scatter(xa[m], xb[m], s=6, color=PALETTE[i % len(PALETTE)], edgecolors='none', alpha=0.9, label=f"C{c} (n={m.sum():,})")
ax.set_xlabel(fa, fontsize=AXIS_PT); ax.set_ylabel(fb, fontsize=AXIS_PT)
ax.tick_params(labelsize=AXIS_PT); ax.set_title(f"Scatter: {fa} vs {fb}", fontsize=AXIS_PT+1)
ax.legend(loc='best', fontsize=AXIS_PT)
fig.savefig(buf_sc, format="png", bbox_inches="tight"); plt.close(fig); buf_sc.seek(0)

# a simple violin for A and B to prove layout
def vbuf(vals_by_group, ylabel, title):
    b = io.BytesIO()
    fig = plt.figure(figsize=(3.0, 2.8), dpi=FIG_DPI); ax = plt.gca()
    ax.violinplot(vals_by_group, showmeans=False, showmedians=True, showextrema=False)
    ax.set_xticks(np.arange(1, len(vals_by_group)+1)); ax.set_xticklabels(list(group_idx), rotation=20, ha='right', fontsize=AXIS_PT)
    ax.set_ylabel(ylabel, fontsize=AXIS_PT); ax.tick_params(labelsize=AXIS_PT); ax.set_title(title, fontsize=AXIS_PT+1)
    lo, hi = np.nanpercentile(np.concatenate(vals_by_group), [1,99]); ax.set_ylim(lo, hi)
    fig.savefig(b, format="png", bbox_inches="tight"); plt.close(fig); b.seek(0)
    return b

groups_a = [xa[idxs] for idxs in group_idx.values()]
groups_b = [xb[idxs] for idxs in group_idx.values()]
buf_va = vbuf(groups_a, fa, f"Violin: {fa} by {OBS_SPLIT_KEY}")
buf_vb = vbuf(groups_b, fb, f"Violin: {fb} by {OBS_SPLIT_KEY}")

# add to CURRENT prs (created in Cell 6)
slide = prs.slides.add_slide(prs.slide_layouts[5])
# reuse your set_title from earlier
set_title(slide, f"{fa} vs {fb}", f"min FDR={rr['min_fdr']:.2e} | max |ρ|={rr['max_abs_rho']:.3f}")

left = Inches(0.3); top = Inches(1.2); img_w = Inches(3.0); img_h = Inches(2.8); gap = Inches(0.2)
slide.shapes.add_picture(buf_sc, left, top, width=img_w, height=img_h)
slide.shapes.add_picture(buf_va, left + img_w + gap, top, width=img_w, height=img_h)
slide.shapes.add_picture(buf_vb, left + 2*(img_w + gap), top, width=img_w, height=img_h)

prs.save(OUTPUT_PPTX)
print("Saved 1-slide smoke test to:", OUTPUT_PPTX)


Saved 1-slide smoke test to: D:\Sam\4i_Liposarcoma_Data_YR2025\LPS_246_Abema_July_2025\standard_adata_sub_sub_pairs_by_treatment.pptx
